# Optimized Multi-Model Recommendation System

This notebook is structured to easily train, tune, and compare multiple machine learning models (e.g., RandomForest, LightGBM, XGBoost) for the doctor recommendation task. 

**To run a different set of models, only the 'Model Configuration' cell needs to be edited.**

### Step 1: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import pgeocode
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import all the model types you want to test
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb

# --- Data Loading ---
# Update these file paths to match your system
file_paths = {
    "patient": "Master_df_sample.xlsx - Patient_df.csv",
    "encounter": "Master_df_sample.xlsx - Encounter_df.csv",
    "provider": "Master_df_sample.xlsx - Provider_df.csv",
    "hospital": "Master_df_sample.xlsx - Hospital_df.csv"
}
patient_df = pd.read_csv(file_paths['patient'])
encounter_df = pd.read_csv(file_paths['encounter'])
provider_df = pd.read_csv(file_paths['provider'])
hospital_df = pd.read_csv(file_paths['hospital'])
print("Dataframes loaded successfully.")

### Step 2: Feature Engineering
(This section remains the same as it's common for all models)

In [ ]:
# Merge all dataframes
master_df = pd.merge(pd.merge(pd.merge(encounter_df, patient_df, on='patient_id'), provider_df, on='provider_id'), hospital_df, left_on='hospital_affiliation', right_on='hospital_id', how='left')

# Create match features
master_df['race_match'] = (master_df['race'] == master_df['provider_race']).astype(int)
master_df['ethnicity_match'] = (master_df['ethnicity'] == master_df['provider_ethnicity']).astype(int)
master_df['language_match'] = (master_df['language_match'] == True).astype(int)

# Create geographic feature
master_df['distance_km'] = pgeocode.GeoDistance('US').query_postal_code(master_df['zip_code'].astype(str).tolist(), master_df['zip_code_hosp'].astype(str).tolist())
mean_dist_by_specialty = master_df.groupby('specialty')['distance_km'].transform('mean')
master_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True); master_df['distance_km'].fillna(master_df['distance_km'].mean(), inplace=True)
min_distance, max_distance = master_df['distance_km'].min(), master_df['distance_km'].max()
master_df['proximity_score'] = 1 - ((master_df['distance_km'] - min_distance) / (max_distance - min_distance))

# Create composite target score
scaler = MinMaxScaler()
master_df[['satisfaction_norm', 'adherence_norm']] = scaler.fit_transform(master_df[['patient_satisfaction', 'treatment_adherence']])
master_df['success_score'] = (master_df['adherence_norm'] * 0.5 + master_df['satisfaction_norm'] * 0.5)

print("Feature engineering complete.")

### Step 3: Model Configuration Block
**This is the main section to edit.** Add, remove, or modify the models and their hyperparameter grids in this dictionary.

In [ ]:
model_configs = {
    'RandomForest': {
        'estimator': RandomForestRegressor(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [50, 100, 150],
            'max_depth': [10, 20, None],
            'min_samples_leaf': [2, 4],
            'max_features': ['sqrt', 'log2']
        }
    },
    'LightGBM': {
        'estimator': lgb.LGBMRegressor(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'num_leaves': [20, 31, 40],
            'max_depth': [-1, 10, 20]
        }
    },
    'XGBoost': {
        'estimator': xgb.XGBRegressor(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.7, 0.8],
            'colsample_bytree': [0.7, 0.8]
        }
    }
}

print(f"Prepared configurations for {list(model_configs.keys())}")

### Step 4: Main Experiment Loop

This loop now iterates through each model and each patient segment, running the full training and validation pipeline automatically.

In [ ]:
# --- Define Features and Target ---
master_df.rename(columns={'cultural_competency_rating_y': 'cultural_competency_rating_prov'}, inplace=True)
features = ['years_experience', 'cultural_competency_rating_prov', 'communication_rating', 'race_match', 'ethnicity_match', 'language_match', 'proximity_score', 'interpreter_services_24_7']
target = 'success_score'

# --- Dictionaries to Store All Results ---
all_results = {}

# --- Main Loop ---
for model_name, config in model_configs.items():
    print(f"\n{'='*20} RUNNING EXPERIMENT FOR MODEL: {model_name.upper()} {'='*20}")
    
    all_results[model_name] = {
        'trained_models': {},
        'test_metrics': {},
        'best_params': {},
        'feature_weights': {}
    }
    
    unique_preferences = master_df['cultural_preferences'].unique()
    for preference in unique_preferences:
        print(f"\n--- Training segment: '{preference}' ---")
        segment_df = master_df[master_df['cultural_preferences'] == preference].copy()
        
        if len(segment_df) < 100: continue
            
        X_segment, y_segment = segment_df[features], segment_df[target]
        X_train_val, X_test, y_train_val, y_test = train_test_split(X_segment, y_segment, test_size=0.2, random_state=42)
        
        random_search = RandomizedSearchCV(
            estimator=config['estimator'],
            param_distributions=config['param_grid'],
            n_iter=20, # Reduced for speed, increase for more thorough search
            cv=3,      # Reduced for speed
            verbose=0, 
            random_state=42, 
            scoring='neg_mean_squared_error'
        )
        
        print(f'Tuning on {len(X_train_val)} samples...')
        random_search.fit(X_train_val, y_train_val)
        best_model = random_search.best_estimator_
        
        print("Evaluating on test set...")
        final_predictions = best_model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
        mae = mean_absolute_error(y_test, final_predictions)
        r2 = r2_score(y_test, final_predictions)
        
        # Store all results for this model and segment
        all_results[model_name]['trained_models'][preference] = best_model
        all_results[model_name]['test_metrics'][preference] = {'rmse': rmse, 'mae': mae, 'r2': r2}
        all_results[model_name]['best_params'][preference] = random_search.best_params_
        all_results[model_name]['feature_weights'][preference] = pd.Series(best_model.feature_importances_, index=features)
        
print("\nAll experiments complete.")

### Step 5: Consolidate and Compare Results

Finally, we can process the `all_results` dictionary to create a single DataFrame that compares the performance (e.g., RMSE) of each model across each patient segment.

In [ ]:
# Create a summary DataFrame for easy comparison
summary_list = []
for model_name, results in all_results.items():
    for preference, metrics in results['test_metrics'].items():
        row = {
            'model_name': model_name,
            'preference_group': preference,
            'rmse': metrics['rmse'],
            'mae': metrics['mae'],
            'r2': metrics['r2']
        }
        summary_list.append(row)

summary_df = pd.DataFrame(summary_list)

print("--- Final Performance Comparison ---")
display(summary_df.sort_values(by=['preference_group', 'rmse']))

# You can also save this summary to a file
# summary_df.to_csv('model_comparison_results.csv', index=False)